<a href="https://colab.research.google.com/github/rayaan27/Fly-Rank-Internship/blob/main/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rayaan27/Fly-Rank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 3: Structured Content Archetype Clustering.**

The question behind this lane is *"What performance archetypes exist across the content
inventory?"* — that's a shape-of-the-population question, not a yes/no question and not a
"which one first" question. There's no observed outcome I'm trying to predict for each page
(no decline event, no click I'm forecasting), so this isn't classification, ranking, or scoring.
It's **clustering**: an unsupervised task where I group ~30k pages into a handful of recurring
"types" (e.g. *champions*, *stale visible pages*, *hidden gems*) based on how their performance
and content signals move together, then hand each group a human-readable profile and an action.

Checked against the skill's mapping table: *"What kinds of items exist?"* → Clustering, target =
none (unsupervised), metric = silhouette + human sense-check. That's exactly this lane.

In [7]:
# --- Environment bootstrap: make sure the starter data is reachable, however this notebook was opened ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rayaan27/Fly-Rank-Internship"
REPO_DIR = "Fly-Rank-Internship"

if IN_COLAB:
    # Opening a notebook straight from a GitHub link only downloads that one file --
    # the repo (and its data/raw/) isn't there yet, so clone it if we don't see it.
    if not os.path.isdir("data/raw") and not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if not os.path.isdir("data/raw") and os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
else:
    # Local/Jupyter: walk up until we find the repo root (has data/raw/)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), (
    "starter CSV not found -- make sure this notebook lives inside the repo "
    "(work/notebooks/) or update REPO_URL above to your own fork."
)
print("Starter data found. You're ready.\n")

# Quick self-check: does my question sound like a clustering question, not a decision-per-page one?
lane_question = "What performance archetypes exist across the content inventory?"
signal_words = ["which one", "will it", "priority", "first"]  # would point to ranking/classification instead
print("Lane question:", lane_question)
print("Contains a 'pick one / predict one' signal word?",
      any(w in lane_question.lower() for w in signal_words))
print("-> No. This is a 'what groups exist' question: clustering.")


Working dir: /Fly-Rank-Internship
Starter data found. You're ready.

Lane question: What performance archetypes exist across the content inventory?
Contains a 'pick one / predict one' signal word? False
-> No. This is a 'what groups exist' question: clustering.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is **no target column** — clustering is unsupervised, which is the point: I'm not
predicting a known outcome, I'm discovering structure that nobody labeled ahead of time.

What I *do* need is a **proxy feature space**: the set of safe, non-leaky signals whose joint
pattern defines an archetype. I'm using per-page performance and content signals — impressions,
clicks, sessions, engagement rate, scroll rate, AI-referred sessions, CTR, average position,
word count, and content age/freshness — all measured over the same trailing-90-day window.

Two things I deliberately leave **out** of the feature space, per the data skill's warning:
- `trend_direction` / `trend_pct` — these define the *decline* label used in Lane 2. Folding
  them in here would mean my "archetypes" are secretly just re-discovering the decline label,
  not finding genuinely new structure.
- `content_id` / `client_id` — pseudonymous identifiers, for grouping only, never features.

The output of clustering *becomes* a new column (a cluster id per page) — that's the closest
thing to a "target" I produce, and I sketch it below in section 4.

In [8]:
# The label trap, made concrete: confirm trend_* columns are excluded from my feature list
excluded_as_features = ["trend_direction", "trend_pct", "content_id", "client_id"]
print("Columns I will NEVER use as clustering features:", excluded_as_features)
print("Reason: trend_* already encodes the Lane-2 decline label; IDs are pseudonyms, not signal.")


Columns I will NEVER use as clustering features: ['trend_direction', 'trend_pct', 'content_id', 'client_id']
Reason: trend_* already encodes the Lane-2 decline label; IDs are pseudonyms, not signal.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Per the framing skill's table, clustering's typical metric is **silhouette score + a human
sense-check** — there's no ground truth to score against, so "good" has to combine a
computable number with a qualitative check:

1. **Silhouette score** (range -1 to 1): how much tighter a page sits inside its own cluster
   than the nearest other cluster, averaged across all pages. I'll treat anything meaningfully
   above 0 as "real structure exists," and compare scores across a few values of *k* (number
   of clusters) to pick one that's both cohesive and not just 2 giant blobs.
2. **Human sense-check**: for each cluster, do the *typical* feature values actually match a
   nameable archetype from the lane guide (champion, hidden gem, stale visible page, etc.), and
   is the cluster a defensible size (not 1 page, not 29,000 pages)?

I can compute silhouette **today**, on a trivial baseline, to prove the metric isn't
hypothetical — I do that right below before building the real feature set in section 4.

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import pandas as pd, numpy as np

# (working directory already resolved in the bootstrap cell above)
_df_peek = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Trivial baseline: cluster on just 2 obvious, always-present numeric columns
baseline_X = _df_peek[["impressions_90d", "clicks_90d"]].fillna(0)
baseline_X_scaled = StandardScaler().fit_transform(baseline_X)
baseline_km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(baseline_X_scaled)
baseline_score = silhouette_score(baseline_X_scaled, baseline_km.labels_)

print(f"Baseline silhouette (2 features, k=3): {baseline_score:.3f}")
print("-> The metric is computable today, on the raw data, before any real feature engineering.")


Baseline silhouette (2 features, k=3): 0.889
-> The metric is computable today, on the raw data, before any real feature engineering.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page).** The starter dataset is 30,000 rows × 44 columns, one row
per pseudonymized page, across 32 clients, all metrics aggregated over a trailing 90-day window.
That's exactly the grain clustering needs here — I'm grouping *pages*, not clients, not
queries, not days.

Below: the raw unit of analysis, then the real (non-leaky) feature set, then a k=6 clustering
run whose output — a per-page `archetype_cluster` column — is the closest thing this lane has
to a "target column." I sketch what that column looks like and give each cluster a first-pass
profile against the lane guide's archetype list.

In [10]:
# --- Unit of analysis: one row = one page ---
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows (pages) x", df.shape[1], "columns")
print("One row =", df.loc[0, "content_id"], "-> a single pseudonymized content item")
display_cols = ["content_id", "client_id", "content_type", "main_intent", "word_count",
                 "impressions_90d", "clicks_90d", "ctr", "avg_position", "engagement_rate"]
df[display_cols].head(5)


30000 rows (pages) x 44 columns
One row = content_304f48230142 -> a single pseudonymized content item


,content_id,client_id,content_type,main_intent,word_count,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,3803,29,0.76,10.6,5.88
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,15320,7,0.05,20.3,0.00
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,12581,11,0.09,36.5,0.00
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,11751,58,0.49,6.2,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,19140,24,0.13,44.0,0.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I could write `if word_count < 500 and impressions_90d > 1000: archetype = "thin visible"` —
and I'd get *one* archetype, badly. The real problem is that "what kind of page is this" depends
on **how ~10 signals move together at once** (impressions, clicks, CTR, position, engagement,
scroll, AI traffic, freshness, word count), and those interactions aren't the same shape for
every client or content type. A rule needs a human to pre-decide every threshold and every
combination up front — with 10 correlated numeric signals that's not a handful of if-statements,
it's an unmanageable decision tree that nobody actually wrote down, and it would silently break
the moment thresholds needed to differ by content type or client scale.

Clustering instead *discovers* which combinations of signals actually co-occur in the data,
without me pre-specifying the rule. Below I show this isn't just a theoretical claim: a
one-metric rule ("split at the median CTR") cuts across most of the multi-signal clusters
rather than reproducing them — evidence that a single threshold isn't standing in for what
the six-signal clusters are picking up on.

In [11]:
# Build the real (non-leaky) feature set, run k=6 clustering, and sketch the "target" column.

feature_candidates = [
    "impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "content_age_days", "days_since_last_update",
]

X = df[feature_candidates].copy()

# Missingness follows content_type (e.g. feedly articles have no word_count) -> add a flag,
# don't silently fillna(0) and pretend "no data" means "zero".
X["has_word_count"] = df["word_count"].notna().astype(int)
X["word_count"] = X["word_count"].fillna(0)

# Heavy-tailed counts -> log1p, same idea as the starter pipeline's log_impressions_90d etc.
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d", "ai_sessions_90d"]:
    X[col] = np.log1p(X[col])

X = X.fillna(0)

X_scaled = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=6, n_init=10, random_state=42)
df["archetype_cluster"] = km.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df["archetype_cluster"])
print(f"Silhouette (real feature set, k=6): {sil:.3f}")
print()
print("Cluster sizes (sanity check for degenerate clusters):")
print(df["archetype_cluster"].value_counts().sort_index())


Silhouette (real feature set, k=6): 0.225

Cluster sizes (sanity check for degenerate clusters):
archetype_cluster
0      160
1    12662
2     5870
3     6797
4     1301
5     3210
Name: count, dtype: int64


In [12]:
# Sketch of what the "target" column looks like: content_id + assigned archetype
df[["content_id", "archetype_cluster"] + feature_candidates[:4]].head(8)


,content_id,archetype_cluster,impressions_90d,clicks_90d,sessions_90d,engaged_sessions_90d
0,content_304f48230142,2,3803,29,17,1
1,content_a1fb4e703a9e,1,15320,7,9,0
2,content_9aa793d4d895,1,12581,11,11,0
3,content_331d6c4de07b,3,11751,58,78,1
4,content_d99b7a2d90ca,2,19140,24,145,0
5,content_d4084a4bc775,1,3970,1,5,0
6,content_9a34b442b552,1,20,0,1,0
7,content_a63219c6e95a,3,1724,1,28,1


In [13]:
# Profile each cluster against the lane guide's archetype vocabulary
profile = (
    df.groupby("archetype_cluster")[
        ["impressions_90d", "clicks_90d", "ctr", "avg_position",
         "engagement_rate", "word_count", "days_since_last_update"]
    ]
    .median()
    .round(2)
)
profile["n_pages"] = df["archetype_cluster"].value_counts().sort_index()
profile


,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,word_count,days_since_last_update,n_pages
archetype_cluster,,,,,,,,
0,3.0,1.0,33.33,3.5,0.00,892.0,20.0,160
1,325.0,0.0,0.00,11.5,0.00,2810.0,20.0,12662
2,8306.0,23.0,0.29,8.8,3.03,3005.0,20.0,5870
3,742.0,1.0,0.03,13.5,0.00,1230.5,22.0,6797
4,10061.0,26.0,0.24,18.0,2.08,5268.0,104.0,1301
5,25.0,0.0,0.00,7.7,0.00,2708.0,20.0,3210


In [14]:
# Does a single-metric rule reproduce these clusters? (evidence for section 5's claim)
df["ctr_median_rule"] = (df["ctr"] > df["ctr"].median()).astype(int)
crosstab = pd.crosstab(df["archetype_cluster"], df["ctr_median_rule"],
                        rownames=["cluster"], colnames=["above_median_ctr_rule"])
print(crosstab)

mixed_clusters = (crosstab.min(axis=1) > 0).sum()
total_clusters = crosstab.shape[0]
print()
print(f"{mixed_clusters}/{total_clusters} clusters contain BOTH above- and below-median-CTR pages.")
print("A single CTR threshold mostly cuts across the multi-signal clusters rather than")
print("reproducing them -- one number isn't standing in for the six-signal archetype.")


above_median_ctr_rule     0     1
cluster                          
0                         0   160
1                      7809  4853
2                       557  5313
3                      3862  2935
4                       287  1014
5                      2709   501

5/6 clusters contain BOTH above- and below-median-CTR pages.
A single CTR threshold mostly cuts across the multi-signal clusters rather than
reproducing them -- one number isn't standing in for the six-signal archetype.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.